# SCAPI Shopper API — Products (search + product by ID)

Mints a **guest shopper token** via SLAS (using the SDK's
`b2c_tooling_sdk.slas.get_guest_token`), then calls two SCAPI **Shopper** APIs
with that bearer token:

- **Shopper Search** — `product-search` (find products by keyword)
- **Shopper Products** — `products/{id}` (fetch one product's details)

The SDK is a *tooling* SDK: it provides the SLAS shopper-token helper but no
dedicated shopper-storefront client, so the calls themselves are plain `httpx`
GETs to the SCAPI Shopper endpoints.

Connection settings come from `../dw.json`; the search/product parameters are
editable in the **Parameters** cell below.

In [ ]:
import json
from pathlib import Path

import httpx

from b2c_tooling_sdk.slas import SlasTokenConfig, get_guest_token

# --- Connection settings (read from dw.json) ---
raw = json.loads(Path("../dw.json").resolve().read_text())
SHORT_CODE = raw["shortCode"]
ORGANIZATION_ID = raw["organizationId"]
SITE_ID = raw["siteId"]
SLAS_CLIENT_ID = raw["slasClientId"]
REDIRECT_URI = raw.get("slasRedirectUri", "http://localhost:3000/callback")

SCAPI_BASE_URL = f"https://{SHORT_CODE}.api.commercecloud.salesforce.com"
print(f"Instance {SHORT_CODE} · org {ORGANIZATION_ID} · site {SITE_ID}")

In [ ]:
# --- Parameters (edit these) ---
SEARCH_QUERY = "shirt"   # product-search keyword
RESULT_LIMIT = 10        # max hits to fetch/print
PRODUCT_ID = None        # None -> use the first search hit; or set a specific product id

In [ ]:
# Mint a guest shopper token (public SLAS client -> PKCE guest flow).
token = await get_guest_token(
    SlasTokenConfig(
        short_code=SHORT_CODE,
        organization_id=ORGANIZATION_ID,
        slas_client_id=SLAS_CLIENT_ID,
        site_id=SITE_ID,
        redirect_uri=REDIRECT_URI,
    )
)
ACCESS_TOKEN = token.access_token
print(f"Guest token minted (customer_id={token.customer_id}, expires_in={token.expires_in}s)")


async def scapi_get(path: str, params: dict | None = None) -> dict:
    """Authenticated GET against a SCAPI Shopper endpoint (siteId added automatically)."""
    query = {"siteId": SITE_ID, **(params or {})}
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
    async with httpx.AsyncClient(timeout=30) as client:
        response = await client.get(f"{SCAPI_BASE_URL}{path}", params=query, headers=headers)
    if response.status_code != 200:
        raise RuntimeError(f"GET {path} -> HTTP {response.status_code}: {response.text[:300]}")
    return response.json()

## 1. Product search (Shopper Search API)

`GET /search/shopper-search/v1/organizations/{org}/product-search?q=...&siteId=...`

In [ ]:
data = await scapi_get(
    f"/search/shopper-search/v1/organizations/{ORGANIZATION_ID}/product-search",
    {"q": SEARCH_QUERY, "limit": RESULT_LIMIT},
)
hits = data.get("hits") or []
print(f'"{SEARCH_QUERY}" -> {data.get("total", 0)} total result(s):')
for hit in hits:
    price = hit.get("price")
    price_str = f'{price} {hit.get("currency", "")}'.strip() if price is not None else "n/a"
    print(f'  • {hit.get("productId", "?"):<22} {hit.get("productName", "(no name)")}  [{price_str}]')
if not hits:
    print("  (no hits — try a different SEARCH_QUERY)")

## 2. Product by ID (Shopper Products API)

`GET /product/shopper-products/v1/organizations/{org}/products/{id}?siteId=...`

Uses `PRODUCT_ID` if set, otherwise the first search hit above.

In [ ]:
product_id = PRODUCT_ID or (hits[0]["productId"] if hits else None)
if not product_id:
    print("No product id available — set PRODUCT_ID or broaden SEARCH_QUERY.")
else:
    product = await scapi_get(f"/product/shopper-products/v1/organizations/{ORGANIZATION_ID}/products/{product_id}")
    print("id       :", product.get("id"))
    print("name     :", product.get("name"))
    print("type     :", product.get("type"))
    print("price    :", product.get("price"), product.get("currency"))
    print("brand    :", product.get("brand"))
    desc = product.get("longDescription") or product.get("shortDescription")
    if desc:
        print("desc     :", str(desc)[:160].strip(), "...")